In [1]:
%%capture
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton
!pip install --no-deps cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

In [3]:
from unsloth import FastLanguageModel
import torch
from google.colab import userdata


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-3-4b-it",
    max_seq_length=2048,
    dtype = None,
    load_in_4bit=True,
    token=userdata.get('HF_ACCESS_TOKEN')
)

==((====))==  Unsloth 2025.11.2: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


In [5]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [31]:
from datasets import load_dataset

raw_data1 = load_dataset("thnhan/VietnameseHealth-QA-dataset", split="train")
raw_data2 = load_dataset("lqkhoi/viet_med_qa", split="train")

In [32]:
raw_data2 = raw_data2.remove_columns('url')
raw_data2 = raw_data2.remove_columns('tags')
print(raw_data2)

Dataset({
    features: ['question', 'answer'],
    num_rows: 10656
})


In [33]:
from datasets import concatenate_datasets

raw_data = concatenate_datasets([raw_data1, raw_data2])
print(raw_data)

Dataset({
    features: ['question', 'answer'],
    num_rows: 27000
})


In [34]:
data = raw_data.train_test_split(test_size=0.1, shuffle=True, seed = 42)
print(data)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 24300
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 2700
    })
})


In [38]:
train_data = data['train']
test_data = data['test']

In [39]:
messages = [
    {"role": "system", "content": "You are a helpfull assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]

formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(formatted_text)

<bos><start_of_turn>user
You are a helpfull assistant.

What is the capital of France?<end_of_turn>
<start_of_turn>model



In [40]:
from unsloth.chat_templates import standardize_data_formats
train_data = standardize_data_formats(train_data)


In [41]:
train_data[0]

{'question': 'Trẻ mắc bệnh thủy đậu mẹ cho con bú nên kiêng ăn gì?',
 'answer': 'Bé bị thủy đậu nên mẹ tiếp tục cho bé bú và không cần ăn kiêng gì.'}

In [42]:
EOS_TOKEN = tokenizer.eos_token

# SYSTEM_PROMPT = "Bạn là một trợ lý y tế tài năng, chuyên cung cấp thông tin tham khảo. Luôn khuyên người dùng nên tham khảo ý kiến của bác sĩ chuyên khoa."

def format_with_system_prompt(example):
  example['conversations'] = [
        {"role": "user", "content": example['question']},
        {"role": "assistant", "content": example['answer']}
  ]
  return example

train_data = train_data.map(format_with_system_prompt)


def formatting_prompts_func(examples):
  convos = examples["conversations"]
  texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
  return { "text" : texts, }

train_data = train_data.map(formatting_prompts_func, batched = True)


Map:   0%|          | 0/24300 [00:00<?, ? examples/s]

Map:   0%|          | 0/24300 [00:00<?, ? examples/s]

In [44]:
train_data[0]['text']

'<start_of_turn>user\nTrẻ mắc bệnh thủy đậu mẹ cho con bú nên kiêng ăn gì?<end_of_turn>\n<start_of_turn>model\nBé bị thủy đậu nên mẹ tiếp tục cho bé bú và không cần ăn kiêng gì.<end_of_turn>\n'

In [45]:
train_data.to_json("train.jsonl", lines = True)
test_data.to_json("test.jsonl", lines = True)

Creating json from Arrow format:   0%|          | 0/25 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

4376911